In [ ]:
from datetime import date

import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine
##preguntas que responde este hecho


# database connections 

In [2]:
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['CO_SA']
    config_etl = config['ETL_PRO']

url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
co_sa = create_engine(url_co)
etl_conn = create_engine(url_etl)

# Extract

In [7]:
hecho_servicio = pd.read_sql_table('mensajeria_servicio', url_co)
tabla_fases = pd.read_sql_table('mensajeria_estadosservicio', url_co)
dim_estado = pd.read_sql_table('dim_estado', etl_conn)
dim_fecha = pd.read_sql_table('dim_fecha', etl_conn)
dim_hora = pd.read_sql_table('dim_hora', etl_conn)
dim_geografia = pd.read_sql_table('dim_geografia', etl_conn)
dim_mensajero = pd.read_sql_table('dim_mensajero', etl_conn)
dim_novedad = pd.read_sql_table('dim_novedad', etl_conn)
dim_sede = pd.read_sql_table('dim_sede', etl_conn)
dim_tipo_servicio = pd.read_sql_table('dim_tipo_servicio', etl_conn)
dim_estado.info()
#select count(*) from mensajeria_documentoasociado; 


<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   key_dim_estado   6 non-null      int64  
 1   id               6 non-null      int64  
 2   nombre_estado    6 non-null      str    
 3   orden_secuencia  5 non-null      float64
dtypes: float64(1), int64(2), str(1)
memory usage: 324.0 bytes


# Transformations

El campo `descripcion` original duplica casi textualmente a `nombre` (ej. 'Recogido por mensajero' / 'Recogido por mensajero'), asi que no aporta y se descarta.

El `Orden_Secuencia` NO se puede tomar de ninguna columna fuente -- se calcula manualmente segun el flujo oficial del documento de diseno:
Iniciado(1) -> Con mensajero Asignado(2) -> Recogido por mensajero(3) -> Entregado en destino(4) -> Terminado completo(5).

'Con novedad' NO forma parte de esa secuencia principal (es un estado de excepcion, no un paso del flujo), por lo que se le asigna orden_secuencia = NaN/None.

In [8]:
hecho_servicio.drop(columns=['descripcion','fecha_solicitud','hora_solicitud','fecha_deseada','hora_deseada', 'nombre_recibe','telefono_recibe','descripcion_pago'], inplace=True)
tabla_fases.drop(columns=['foto','observaciones','es_prueba','foto_binary'], inplace=True)
hecho_servicio.head()

,id,nombre_solicitante,ida_y_regreso,activo,novedades,cliente_id,destino_id,mensajero_id,origen_id,tipo_pago_id,...,ciudad_origen_id,hora_visto_por_mensajero,visto_por_mensajero,descripcion_multiples_origenes,mensajero2_id,mensajero3_id,multiples_origenes,asignar_mensajero,es_prueba,descripcion_cancelado
0,34,chat_GPT,False,False,,5,214,NaN,241,2,...,1,None,,,NaN,NaN,False,True,False,
1,35,chat_GPT,False,True,,5,214,7.0,236,2,...,1,None,,,NaN,NaN,False,False,True,
2,36,chat_GPT,False,False,,5,214,NaN,236,2,...,1,None,,,NaN,NaN,False,True,False,
3,41,chat_GPT,False,False,,5,214,NaN,241,1,...,1,None,,,NaN,NaN,False,True,False,
4,42,chat_GPT,False,False,,5,214,NaN,241,1,...,1,None,,,NaN,NaN,False,True,False,


In [5]:
# TODO: si tus companeros usan otros nombres de estado en su BD, ajusta este diccionario.
orden_secuencia_map = {
    'Iniciado': 1,
    'Con mensajero Asignado': 2,
    'Recogido por mensajero': 3,
    'Entregado en destino': 4,
    'Terminado completo': 5,
    'Con novedad': None,  # estado de excepcion, fuera del flujo principal
}

dim_estado['orden_secuencia'] = dim_estado['nombre'].map(orden_secuencia_map)

# Verificar que ningun estado quedo sin mapear por error de tipeo
sin_mapear = dim_estado[~dim_estado['nombre'].isin(orden_secuencia_map.keys())]
if not sin_mapear.empty:
    print('ATENCION: hay estados sin mapear, revisar nombres exactos:')
    print(sin_mapear)

dim_estado

,id,nombre,orden_secuencia
0,4,Recogido por mensajero,3.0
1,5,Entregado en destino,4.0
2,3,Con novedad,NaN
3,6,Terminado completo,5.0
4,1,Iniciado,1.0
5,2,Con mensajero Asignado,2.0


# Alinear nombres de columnas al diagrama
ID_Estado (PK) -> indice al cargar | Nombre_Estado | Orden_Secuencia

In [6]:
dim_estado.rename(columns={
    'nombre': 'nombre_estado',
}, inplace=True)
dim_estado['saved'] = date.today()
dim_estado

,id,nombre_estado,orden_secuencia,saved
0,4,Recogido por mensajero,3.0,2026-06-23
1,5,Entregado en destino,4.0,2026-06-23
2,3,Con novedad,NaN,2026-06-23
3,6,Terminado completo,5.0,2026-06-23
4,1,Iniciado,1.0,2026-06-23
5,2,Con mensajero Asignado,2.0,2026-06-23


# load

In [7]:
# NOTA: cambiado de if_exists='replace' a 'append' porque la tabla ahora se crea por DDL
# (ver sqlscripts.yml) con su PK real. 'replace' borraria esa tabla y la reemplazaria sin
# restricciones. Antes de correr este notebook, asegurate que main.py ya ejecuto el DDL
# (o ejecuta el script de sqlscripts.yml manualmente si la tabla aun no existe).
dim_estado.to_sql('dim_estado', etl_conn, if_exists='append', index=False)

6